# Streaming VAE Anomaly Detection — GPU Demo

Clones the repo (the mini real-ERA5 dataset ships with it, no download needed) and runs the MLP-VAE streaming anomaly detector end to end on a free Colab GPU.

Runtime menu → Change runtime type → GPU, then run all cells.

**Setup:** this is a private repo, so before running you need a GitHub token as a Colab secret — key icon in the left sidebar -> new secret named `GITHUB_TOKEN`, value = a token from [github.com/settings/tokens](https://github.com/settings/tokens) (read-only `repo` access is enough), then toggle "Notebook access" on.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
import os

REPO_URL = "https://github.com/hadasecohen/streaming-vae-anomaly-detection.git"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Private repo: add a GitHub token as a Colab secret first --
    # key icon in the left sidebar -> Secrets -> new secret named GITHUB_TOKEN,
    # value = a token from github.com/settings/tokens (read-only "repo" access is enough)
    # -> toggle "Notebook access" on.
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    clone_url = REPO_URL.replace("https://", f"https://{token}@")
    if not os.path.exists("repo"):
        !git clone $clone_url repo
    %cd repo
elif not os.path.exists("run_regression.py"):
    # Already inside a local checkout (e.g. running from notebooks/cases/) --
    # move to the repo root instead of cloning a redundant nested copy.
    %cd ../..

!pip install -q -r requirements.txt


In [ ]:
# Runs offline warmup training + online streaming anomaly detection.
# Swap MLP -> LSTM / TF_VAE, or group -> clean / point / contextual, for the other 11 combinations.
!python -m experiments.era5.MLP.group.run_era5_group_MLP_standalone

In [ ]:
# The METRICS / THRESHOLD TUNING block at the end of run.log is the run's own summary
with open("runs/ERA5-group/MLP/standalone/run.log") as f:
    lines = f.readlines()
print("".join(lines[-40:]))

In [ ]:
# Quick look at the injected anomalies vs the real signal for one feature (t2m)
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/era5/mini_50pct/era5_group_anomalies.csv", parse_dates=["valid_time"], index_col="valid_time")
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(df.index, df["t2m"], lw=0.8, label="t2m")
anom = df[df["is_anomaly"] == 1]
ax.scatter(anom.index, anom["t2m"], color="red", s=10, label="injected anomaly", zorder=3)
ax.axvline(df.index[35064], color="gray", ls="--", lw=1, label="warmup / test split")
ax.legend()
ax.set_title("Real ERA5 series (group scenario) — t2m")
plt.show()